# Imports and data:

In [1]:
import pandas as pd
import regex as re

In [2]:
w1 = pd.read_excel("data/WINNERS 2000-2010 - 05.xlsx")
w2 = pd.read_excel("data/WINNERS 2011-2020 - 05.xlsx")
w3 = pd.read_excel("data/WINNERS 2021-2025 - 05.xlsx")
w4 = pd.read_excel("data/WINNERS 2026-2030 - 05.xlsx")

In [3]:
winners = pd.concat([w1, w2, w3, w4])

winners.shape

(43457, 10)

In [4]:
winners.dtypes

Name                   object
School                 object
Year                    int64
City                   object
Award                  object
Division               object
Category               object
School / Company       object
Division / Position    object
Dance Category         object
dtype: object

In [5]:
winners

,Name,School,Year,City,Award,Division,Category,School / Company,Division / Position,Dance Category
0,NaN,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
1,NaN,"The Kirov Academy, DC",2000,"Washington, DC",Outstanding School Award,NaN,Special Awards,NaN,NaN,NaN
2,8+3,Inwood Dance Company,2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
3,Cabaret,"Dance Conservatory, DE",2000,"Washington, DC",3rd Place (tie),NaN,Ensembles,NaN,NaN,NaN
4,Make Them Hear You,"Allegro Dance Arts Academy, NJ, USA",2000,"Washington, DC",2nd Place,NaN,Ensembles,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
8296,Dance For Six,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
8297,Coppelia Friends Dance,NaN,2026,"San Francisco, CA (Mar)",Top 12,NaN,NaN,"Westlake School for the Performing Arts, CA",Large Ensembles,NaN
8298,Kira Fargas,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN
8299,Hadassah Perry,NaN,2026,"San Francisco, CA (Mar)",Outstanding Choreographer Award,NaN,NaN,For All The Work,Special Awards,NaN


In [6]:
winners.isna().sum()

Name                     554
School                 36132
Year                       0
City                       0
Award                    174
Division               36913
Category               36477
School / Company        8286
Division / Position    11327
Dance Category         10780
dtype: int64

- Some awards are for the school overall rather than a person
- Some awards are for the name of the dance rather than people
- School awards wouldn't have a Dance Category or Position, which would explain some entries being null
- NaNs in certain columns can actually tell you what kind of award is being given:
    - NaN in School/Company ; School Award
- Some people don't belong in a School/Company

# Data Cleanup:

- Inconsistent patterns for recording names:
    - First Name Last Name(s)
    - Last Name, First Name
    - First Name Last Name, [US state] (sometimes followed by what US state they're from) 
    - First Name Middle Name(s) and Last Name(s)
    - [1st person] & [2nd person]
    - [1st person], [2nd person]
    - [1st person] (age), [2nd person] (age)
- Removing hanging whitespace from all string columns since it will effect analysis
- Turn everything uppercase, since there are inconsistent capitalization (or lack thereof)

In [7]:
winners = winners.apply(lambda col: col.str.upper() if col.dtype == "object" else col)

In [8]:
def split_2_names(string):
    if pd.isna(string):
        return string  # keep NaN
    
    if isinstance(string, str):
        return [s.strip() for s in string.split("&")]
    
    return string

winners.Name = winners.Name.apply(split_2_names)
winners = winners.explode("Name", ignore_index=True)

In [9]:
winners["Name"] = winners.Name.str.strip()

In [10]:
names = winners.Name.dropna().unique()

with open("yagp_names.txt", "w") as file:
    file.writelines(names + "\n")

## Separating awards for schools and people:

In [11]:
# school_awards = winners[
#     (winners.Name.isna()) & 
#     (winners.School.isna() == False) &
#     (winners.Award.isna() == False)
# ]

# school_awards

In [12]:
# dancers = winners[
#     (winners.Name.isna() == False) &
#     (winners.Name.isna() == False) & 
# ]

# Interesting Aggregates:

In [14]:
winners.Name.value_counts()

Name
FACULTY                         43
CAITLYN DIONEDA                 36
KATIA ALMAYEVA                  33
PAS DE DEUX FROM LE CORSAIRE    31
AURORA KELLOGG                  30
                                ..
DARK ROAD                        1
RUNNING                          1
FLIPBOOK                         1
RHYTHMS OF TEA                   1
DARLI NOGARR                     1
Name: count, Length: 16690, dtype: int64

In [15]:
winners.School.value_counts()

School
SOUTHLAND BALLET ACADEMY                               132
BALLET ACADEMY OF TEXAS, TX, USA                       112
INTERNATIONAL BALLET SCHOOL, CO                         91
ORLANDO BALLET SCHOOL                                   85
THE ROCK SCHOOL FOR DANCE EDUCATION, PA                 83
                                                      ... 
CENTRAL PENNSYLVANIA YOUTH BALLET, PENNSYLVANIA/USA      1
LOS ANGELES BALLET ACADEMY, USA                          1
SOMENTO ARTISTICO CORDOBAS, MEXICO                       1
CALIFORNIA BALLET SCHOOL, CALIFORNIA/USA                 1
BALLET MEXICO, MEXICO                                    1
Name: count, Length: 1758, dtype: int64

In [16]:
winners.groupby(["School", "Award"])["Year"].count()

School                                            Award             
1 DANCE ACADEMY                                   TOP 12                 1
A TOUCH OF CLASS PERFORMING ARTS.                 TOP 12                 1
ACADEMIA CRISTINA CARÁ, BRAZIL                    SILVER MEDAL (TIE)     1
                                                  TOP 12                 2
ACADEMIA DE BALLET CLASICO NINA NOVAK, VENEZUELA  SILVER MEDAL (TIE)     1
                                                                        ..
ZAMUEL BALLET SCHOOL, CO, USA                     3RD PLACE (TIE)        2
                                                  TOP 12                39
                                                  TOP 6                  1
ZEAL STUDIOS                                      TOP 12                 1
                                                  TOP 14                 1
Name: Year, Length: 3366, dtype: int64

In [17]:
winners.groupby(["Year", "School", "Award"])[["Year"]].count()

Year
Year School                                   Award                                
2000 ACADEMY OF DANCE ARTS, IL, USA           1ST PLACE                           3
                                              2ND PLACE                           1
                                              3RD PLACE (TIE)                     2
                                              BRONZE MEDAL (TIE)                  1
                                              OUTSTANDING CHOREOGRAPHER AWARD     1
...                                                                             ...
2010 WORLD BALLET, INC., FL                   TOP 12                              1
     YURI GRIGORIEV SCHOOL OF BALLET, CA, USA TOP 12                              1
     ZAMUEL BALLET SCHOOL, CO, USA            2ND PLACE                           1
                                              3RD PLACE                           1
                                              TOP 12                              9

[4190 rows x 1 columns]